# 235 - Graded decomposition (convex NMF)

**Figures F3-F5.** Every electrode gets a weight on every component instead of one label.

Convex NMF because it accepts **signed** data - high-gamma in baseline-relative dB goes negative, which ordinary NMF cannot take. Same method as Hamilton 2018/2021, Kurteff 2024 and Norman-Haignere 2022 on this data type.

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, "functions")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import lf_decompose as D

RUN = Path("outputs/clustering/kmeans/concat_hg/runs/20260803_175417")
OUT = Path("outputs/clustering/decomposition"); OUT.mkdir(parents=True, exist_ok=True)

X    = np.load(RUN / "X_train.npy").astype(np.float64)
lab  = pd.read_csv(RUN / "labels.csv")
CCOL = next(c for c in lab.columns if c.startswith("cluster_") and not c.endswith("_ranked"))
y5   = lab[CCOL].to_numpy()
pat  = lab["patient_id"].astype(str).to_numpy()

# fsaverage coordinates, for anatomical coherence
_co = pd.read_csv("outputs/250_recon/fsaverage/coords/ALL_PATIENTS_contacts_fsaverage.csv")
_nz = lambda s: str(s).replace("_", "").replace("-", "").upper()
_co["key"] = [f"{p}|{_nz(x)}" for p, x in zip(_co["patient"], _co["name"])]
lab["key"] = [f"{p}|{_nz(e)}" for p, e in zip(lab["patient_id"], lab["electrode"])]
XYZ = lab.merge(_co[["key", "x", "y", "z"]], on="key", how="left")[["x", "y", "z"]].to_numpy()

CONDS = ["audio", "picture", "reading"]; NT = X.shape[1] // 3
print(f"{X.shape[0]} electrodes x {X.shape[1]} features | {len(np.unique(pat))} patients "
      f"| {np.isnan(XYZ).any(1).sum()} without coordinates")

## 1 - How many components? Held-out reconstruction

Fit without a fold of electrodes, project that fold onto the fitted components, measure the error. Unlike silhouette this **can get worse** when the rank is too high, so it is a real criterion (Norman-Haignere 2022).

*Slow - a few minutes. Narrow `KS` while exploring.*

In [ ]:
Xs = D.unit_norm(X)
KS = [2, 3, 4, 5, 6, 8, 10, 12, 14,16,18]
cv = D.cv_rank_curve(Xs, KS, n_folds=5, n_iter=150)
g  = cv.groupby("k")["var_explained"].agg(["mean", "std"])
display(g.round(4))

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.errorbar(g.index, g["mean"], yerr=g["std"], marker="o", capsize=3)
ax.set_xlabel("components"); ax.set_ylabel("held-out variance explained")
ax.set_title("F3a - rank chosen on data the fit never saw", loc="left")
ax.spines[["top", "right"]].set_visible(False)
fig.savefig(OUT / "F3a_cv_rank.png", dpi=150, bbox_inches="tight"); plt.show()

## 2 - The components

Set `K` from the curve above. Each component is a weighted average of real electrodes, so it is a response profile in dB that you can read directly.

In [ ]:
K = 12                      # <- set from the curve above
W, G, C = D.convex_nmf(Xs, K, random_state=0, n_iter=300)
ve = 1 - ((Xs - D.reconstruct(Xs, W, G)) ** 2).sum() / (Xs ** 2).sum()
print(f"in-sample variance explained: {ve:.3f}")

t = np.linspace(0, 100, NT)
fig, axes = plt.subplots(K, 3, figsize=(12, 2.1 * K), sharex=True, sharey=True)
for j in range(K):
    for b, cond in enumerate(CONDS):
        a = axes[j, b]
        a.plot(t, C[j].reshape(3, NT)[b], lw=1.6, color=f"C{j}")
        a.axvline(50, color="0.7", lw=.8, ls=":")     # GO cue
        a.axhline(0, color="0.85", lw=.8)
        if j == 0: a.set_title(cond, fontsize=10)
        if b == 0: a.set_ylabel(f"component {j}", fontsize=9)
        a.spines[["top", "right"]].set_visible(False)
axes[-1, 1].set_xlabel("% of warped trial (50 = GO cue)")
fig.suptitle("F3b - component response profiles", x=.09, ha="left")
fig.tight_layout(); fig.savefig(OUT / "F3b_components.png", dpi=150, bbox_inches="tight"); plt.show()

## 3 - Electrodes are mixtures, not members

If most electrodes sit near a corner the discrete story was fine after all; if most sit in the middle it was not.

In [ ]:
Gn = G / np.maximum(G.sum(1, keepdims=True), 1e-12)
print(json.dumps(D.mixture_summary(Gn), indent=2))

top = Gn.max(1)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(top, bins=40, color="#4a6fa5")
for v, lb in ((0.5, "no majority"), (0.8, "dominated")):
    ax.axvline(v, color="#c1121f", ls="--", lw=1.2)
    ax.text(v, ax.get_ylim()[1] * .92, lb, fontsize=8, rotation=90, ha="right")
ax.set_xlabel("largest component weight per electrode"); ax.set_ylabel("electrodes")
ax.set_title("F4 - how much does the leading component actually lead?", loc="left")
ax.spines[["top", "right"]].set_visible(False)
fig.savefig(OUT / "F4_mixture.png", dpi=150, bbox_inches="tight"); plt.show()

## 4 - Anatomy, coloured honestly

**Hue = leading component, saturation = how much it leads.** Contested cortex desaturates toward grey instead of being handed to an arbitrary winner - the whole point when a large share of electrodes have no majority component.

In [ ]:
PALETTE = plt.get_cmap("tab20").colors[:K]
rgb = D.soft_rgb(Gn, PALETTE)
ok  = ~np.isnan(XYZ).any(1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for a, (i, j, ti) in zip(axes, [(0, 1, "axial (x, y)"), (0, 2, "sagittal (x, z)"),
                                (1, 2, "coronal (y, z)")]):
    a.scatter(XYZ[ok, i], XYZ[ok, j], c=rgb[ok], s=16, lw=0)
    a.set_title(ti, fontsize=10); a.set_aspect("equal"); a.axis("off")
fig.suptitle("F5 - leading component (hue) and confidence (saturation)", x=.02, ha="left")
fig.tight_layout(); fig.savefig(OUT / "F5_soft_anatomy.png", dpi=150, bbox_inches="tight"); plt.show()

obs, ratio = D.spatial_coherence(Gn.argmax(1), XYZ)
print(f"anatomical coherence of the hard argmax, for reference: {obs:.3f} = {ratio:.2f}x chance")

## 5 - Which components share cortex?

Correlation between component loading maps. Positive pairs occupy the same tissue - exactly where a hard label is arbitrary.

In [ ]:
names = [f"c{j}" for j in range(K)]
Cm = pd.DataFrame(np.corrcoef(Gn[ok].T), index=names, columns=names)
display(Cm.round(2))
iu = np.triu_indices(K, 1)
pairs = sorted(zip(Cm.to_numpy()[iu], zip(*iu)))
print(f"most co-located: c{pairs[-1][1][0]} & c{pairs[-1][1][1]}  r={pairs[-1][0]:+.2f}")

np.save(OUT / "G_loadings.npy", G); np.save(OUT / "components.npy", C)
lab.assign(**{f"w{j}": Gn[:, j] for j in range(K)}).to_csv(OUT / "electrode_loadings.csv", index=False)
print("saved ->", OUT)